In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt

import sys
sys.path.insert(0, "../")
from data import get_data, get_site_ids, get_global_grid

In [ ]:
from statsmodels.tsa.stattools import acf
site_uid = "WQS0071"

# uncomment to create 
# rain and water autocorrelation
# plots
"""
lags = 90
def acf_plot(site_uid, ax):
    data = get_data(site_uid=site_uid)
    water = data.water.resample("1D").max()["nitrate_con"]
    rain = data.rain.groupby(["date"])[["precip_in_1d"]].sum().reset_index().set_index("date")
    sm.graphics.tsa.plot_acf(
        water.values, 
        lags=lags,ax=ax, 
        missing="conservative", 
        label="water_nitrate",
        color="tab:orange",
        vlines_kwargs={"colors": "tab:orange"},)
    sm.graphics.tsa.plot_acf(
        rain.values, 
        lags=lags,ax=ax, 
        missing="conservative", 
        label="rain",
        color="tab:blue",
        vlines_kwargs={"colors": "tab:blue"},)
    ax.set_title(f"Autocorrelation {site_uid}")
    ax.set_ylim(-1.1, 1.1)
    ax.set_ylim(-1.1, 1.1)

fig, axes = plt.subplots(17, 5, figsize=(40, 80))
axes = axes.flatten()
for i,site in enumerate(get_site_ids()):
    if i % 5 ==0:
        print(f"{i} sites processed")
    acf_plot(site, axes[i])
    
plt.show()"""

In [ ]:
import statsmodels.tsa.api as sm
from statsforecast.models import AutoARIMA
from statsforecast.arima import arima_string

site_uid = "WQS0048"
data = get_data(site_uid=site_uid)
water = data.water["nitrate_con"].resample("1D").max().dropna()
rain = data.rain.groupby(["date"])[["precip_in_1d"]].sum().reset_index().set_index("date")
print(rain.values)
m = AutoARIMA().fit(rain.values[:,0])
print(arima_string(m.model_))

In [ ]:
global_grid = get_global_grid()

global_grid[global_grid.contained_in_sites.map(len) >= 6]

In [ ]:
import sys
sys.path.insert(0, "../")
from data import get_data, make_site_df

df = make_site_df("WQS0039")

In [ ]:
rain = get_data("WQS0039").rain
rain.columns

In [ ]:
grid = get_data("WQS0039").grid
grid.columns

In [ ]:
glgrid = get_global_grid()
glgrid.columns

In [ ]:
data = get_data("WQS0039")
grid = data.grid
basin = data.basin

basin.head()

In [ ]:
from data.water import get_metadata

df = get_metadata()
df[df.site_uid == "WQS0039"][["longitude", "latitude"]].values[0].tolist()

In [ ]:
from data import weather
w = weather.get_weather("WQS0115")
w.columns

In [ ]:
from data import splits

In [ ]:
folds = splits.make_folds()
f = 3
test_sites = folds.index[folds.fold == f].tolist()
train_sites = folds.index[folds.fold != f].tolist()

print(len(test_sites))
print(len(train_sites))

audit = splits.audit_split(train_sites, test_sites)

assert audit.query("severity == 'hard'").empty

In [ ]:
from data.helper import make_site_df

df = make_site_df("WQS0039")
df.columns

In [ ]:
from data import get_grid

_DEFAULT_DIST_EDGES_M = (50_000, 150_000)

def bucket_map(site_uid, edges=_DEFAULT_DIST_EDGES_M):
    """f: node_id -> dist_bucket, from the grid's dist_to_sensor (fixed bins)."""
    grid = get_grid(site_uid=site_uid)
    e = [-np.inf, *edges, np.inf]
    b = pd.cut(grid["dist_to_sensor"], bins=e, labels=list(range(len(edges)+1)))
    return pd.Series(b.values, index=grid["node_id"], name="bucket")

bucket = bucket_map("WQS0115")
bucket

In [ ]:
from data import get_grid, get_data
from data.features import agg_site_by_bucket, flatten_grid

uid = "WQS0039"
cb, sb, wb = agg_site_by_bucket(uid)

merged = flatten_grid(site_uid=uid)

In [ ]:
print(merged[merged.bucket == 0].head())
print(merged[merged.bucket == 1].head())

print(get_data(site_uid=uid).water.head(10))

In [ ]:
from data import get_data
from data.features import site_df

# hyperparameters
EDGES_M = (50_000, 150_000)
VEL = 1.0
WINDOW = "31"
CENTER_WINDOW = False

uid = "WQS0039"

df = site_df(uid)

In [ ]:
print(df.shape[0])
print(get_data(uid).water.resample("1D").max().shape[0])

print(df.head())

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt

import sys
sys.path.insert(0, "../")

from data import get_data, get_grid
from data.features import site_df, agg_by_bucket, agg_site_to_bucket, _sensible_agg_dicts, daily_nitrate

_DEFAULT_DIST_EDGES_M = (50_000, 150_000)

# hyperparameters
EDGES_M = (50_000, 150_000)
VEL = 1.0
WINDOW = "31"
CENTER_WINDOW = False

def _bucket_map(site_uid, edges=_DEFAULT_DIST_EDGES_M):
    """f: node_id -> dist_bucket, from the grid's dist_to_sensor (fixed bins)."""
    grid = get_grid(site_uid=site_uid)
    e = [-np.inf, *edges, np.inf]
    b = pd.cut(grid["dist_to_sensor"], bins=e, labels=list(range(len(edges) + 1)))
    return pd.Series(b.values, index=grid["node_id"], name="bucket")

def bucket_lags(site_uid, v=1.0, edges=_DEFAULT_DIST_EDGES_M, max_lag_days=None):
    grid = get_grid(site_uid)
    b = grid["node_id"].map(_bucket_map(site_uid, edges))
    med = grid["dist_to_sensor"].groupby(b).median()
    lag = (med / (v * 86400)).round().astype(int)  # days = metres / (m/s * s/day)
    return lag.clip(upper=max_lag_days) if max_lag_days else lag  # Series: bucket -> lag_days

def lag_buckets(weather_b, lags, cols=None, date_col="date", bucket_col="bucket"):
    if cols is None:
        cols = [c for c in weather_b.columns if c not in (date_col, bucket_col)]
    parts = []
    for b, sub in weather_b.groupby(bucket_col, observed=True):
        sub = sub.sort_values(date_col).set_index(date_col).asfreq("D")
        sub[cols] = sub[cols].shift(int(lags.get(b, 0)))  # row t <- value from t-lag
        sub[bucket_col] = b
        parts.append(sub.reset_index())
    return pd.concat(parts, ignore_index=True)

def flatten_buckets(df, fill=False, bucket_col="bucket", value_cols=None):
    if value_cols is None:
        value_cols = [c for c in df.columns if c not in ("date", "bucket", "year")]

    wide = df.pivot(index="date", columns="bucket", values=value_cols)
    wide.columns = [f"{col}_b{int(b)}" for col, b in wide.columns]
    wide = wide.reset_index()
    return wide

def site_df(site_uid):
    BUCKETS = [50_000, 100_000, 150_000]
    VEL = 1.0
    cb, sb, wb = agg_site_by_bucket(site_uid, edges=BUCKETS)
    
    # lag the weather
    lags = bucket_lags(site_uid=site_uid, water_velocity=VEL)
    wb_lag = lag_buckets(weather_b=wb, lags=lags, date_col="date")
    
    # flatten
    c_wide = flatten_buckets(df=cb)
    s_wide = flatten_buckets(df=sb)
    w_wide = flatten_buckets(df=wb)
    
    # get target
    target = daily_nitrate(site_uid=site_uid, agg_meth="max")